# Chapter 13 – Worked examples

This notebook runs the worked example of [Chapter 13](../index.md) live. The code lives in the modules next to it
(`coupler_splits.py`, `reassignment_distance.py`, `ring_splits.py`, `four_substations.py`); the cells only call
them, so a change in a module is a change here too. The ToOp part (`station_toop.py`) runs in `lab_toop.ipynb`.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import coupler_splits
import four_substations
import numpy as np
import reassignment_distance
import ring_splits

## Coupler states that split a chain of four busbars

Three closed couplers join busbars $b_0 \dots b_3$ in a chain. Of the $2^3 = 8$ coupler states, the ones that
leave exactly two connected groups are the three states with exactly one coupler open.

In [ ]:
for is_open, parts in coupler_splits.two_node_configurations(coupler_splits.COUPLERS, 4):
    print('open:', [c for c, o in zip(coupler_splits.COUPLERS, is_open, strict=True) if o], ' groups:', parts)

## From coupler states to candidates and to a reassignment distance

Six line bays sit on the busbars (`ASSET_BUSBAR`). Each split becomes a candidate: two Boolean rows (bus A, bus
B) over the assets. For a target $t$ (True = on bus B) the distance is
$\min(\|a_A \oplus \bar t\| + \|a_B \oplus t\|,\ \|a_A \oplus t\| + \|a_B \oplus \bar t\|)$.

In [ ]:
cands = reassignment_distance.candidates()
for k, cand in enumerate(cands):
    print(k, reassignment_distance.describe(cand))
target = np.array([False, False, False, True, True, True])
print('distances to "L3 L4 L5 on B":', reassignment_distance.reassignment_distance(cands, target))
print('the same split, buses named the other way round, no swap:')
print(reassignment_distance.reassignment_distance(cands, ~target, allow_swap=False))

## A ring instead of a chain

A ring of $n$ busbars needs exactly two open couplers to fall into two groups: $\binom{n}{2}$ states out of $2^n$.

In [ ]:
for n in range(3, 9):
    print(n, ring_splits.count_two_component_states(n), ring_splits.closed_form(n))

## A node-breaker station in pypowsybl

Voltage level S1VL2 of the four-substations example is a double busbar with one coupler: 11 breakers and 22
disconnectors, classified by their role.

In [ ]:
net = four_substations.network()
four_substations.classify_switches(net).groupby(['kind', 'role']).size()